In [16]:
install.packages("e1071")
install.packages("caret")
install.packages("dplyr")
install.packages("naivebayes")


Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



In [17]:
install.packages("ggplot2")
install.packages("rpart")
install.packages("rpart.plot")
install.packages("randomForest")

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



In [6]:
# ====================================================
# Comparar Naive Bayes con y sin Validación Cruzada
# ====================================================

# Cargar librerías necesarias
library(e1071)      # Para naiveBayes
library(caret)      # Para RMSE, validación cruzada y otros
library(dplyr)      # Para manipulación de datos

# Cargar datos preprocesados
train_data <- read.csv("train_preprocessed.csv", stringsAsFactors = TRUE)
test_data  <- read.csv("test_preprocessed.csv", stringsAsFactors = TRUE)

factor_vars <- names(train_data)[sapply(train_data, is.factor)]
for (var in factor_vars) {
  if (var %in% names(test_data)) {
    test_data[[var]] <- factor(test_data[[var]], levels = levels(train_data[[var]]))
  }
}

# Eliminar filas con NA en ambos conjuntos
train_data <- train_data[complete.cases(train_data), ]
test_data  <- test_data[complete.cases(test_data), ]

# Definir los predictores: todas las variables excepto "SalePrice"
predictors <- setdiff(names(train_data), "SalePrice")

# Definir los bins para la variable "SalePrice" (discretización)
n_bins <- 50
unique_vals <- length(unique(train_data$SalePrice))
n_bins <- min(n_bins, unique_vals - 1)
bins <- quantile(train_data$SalePrice, probs = seq(0, 1, length.out = n_bins + 1), na.rm = TRUE)
bins <- unique(bins)  # Evitar cortes repetidos
train_data$SalesPrice_bin <- cut(train_data$SalePrice, breaks = bins, include.lowest = TRUE, dig.lab = 10)
bin_centers <- (head(bins, -1) + tail(bins, -1)) / 2

# ====================================================
# Versión sin Validación Cruzada
# ====================================================

# Entrenar el modelo Naive Bayes sin validación cruzada
nb_model_no_cv <- naiveBayes(SalesPrice_bin ~ ., data = train_data[, c(predictors, "SalesPrice_bin")])

# Predecir en el conjunto de prueba sin validación cruzada
nb_pred_probs_no_cv <- predict(nb_model_no_cv, newdata = test_data[, predictors], type = "raw")

# Calcular las predicciones como valor esperado
nb_pred_no_cv <- apply(nb_pred_probs_no_cv, 1, function(prob_vec) sum(prob_vec * bin_centers))

# Calcular el RMSE para el modelo sin validación cruzada
rmse_no_cv <- RMSE(nb_pred_no_cv, test_data$SalePrice)
cat("RMSE sin validación cruzada:", rmse_no_cv, "\n")

# ====================================================
# Versión con Validación Cruzada
# ====================================================

control_cv <- trainControl(method = "cv", number = 10)

# Entrenar el modelo Naive Bayes con validación cruzada
grid <- expand.grid(laplace = c(0, 1, 2),
                    usekernel = c(TRUE, FALSE),
                    adjust = c(0.5, 1, 2))

results_cv <- data.frame(laplace = numeric(),
                         usekernel = logical(),
                         adjust = numeric(),
                         RMSE = numeric())

for (i in 1:nrow(grid)) {
  params <- grid[i, ]
  cat("Evaluando con validación cruzada: laplace =", params$laplace,
      " usekernel =", params$usekernel,
      " adjust =", params$adjust, "\n")

  nb_model_cv <- train(SalesPrice_bin ~ .,
                       data = train_data[, c(predictors, "SalesPrice_bin")],
                       method = "naive_bayes",
                       trControl = control_cv,
                       tuneGrid = params)

  # Predecir en el conjunto de prueba con el modelo de validación cruzada
  nb_pred_probs_cv <- predict(nb_model_cv, newdata = test_data[, predictors], type = "prob")

  nb_pred_cv <- apply(nb_pred_probs_cv, 1, function(prob_vec) sum(prob_vec * bin_centers))

  # Calcular el RMSE para esta combinación
  rmse_cv <- RMSE(nb_pred_cv, test_data$SalePrice)
  cat("RMSE con validación cruzada:", rmse_cv, "\n")

  # Almacenar el resultado
  results_cv <- rbind(results_cv, cbind(params, RMSE = rmse_cv))
}

# Mostrar los resultados ordenados por RMSE de la validación cruzada
results_cv <- results_cv[order(results_cv$RMSE), ]
cat("Resultados de la búsqueda de hiperparámetros con validación cruzada:\n")
print(results_cv)

# ====================================================
# Comparación de Resultados
# ====================================================

cat("\nComparación de RMSE:\n")
cat("RMSE sin validación cruzada:", rmse_no_cv, "\n")
cat("Mejor RMSE con validación cruzada:", min(results_cv$RMSE), "\n")


RMSE sin validación cruzada: 0.6245234 
Evaluando con validación cruzada: laplace = 0  usekernel = TRUE  adjust = 0.5 
RMSE con validación cruzada: 0.6231303 
Evaluando con validación cruzada: laplace = 1  usekernel = TRUE  adjust = 0.5 
RMSE con validación cruzada: 0.6231303 
Evaluando con validación cruzada: laplace = 2  usekernel = TRUE  adjust = 0.5 
RMSE con validación cruzada: 0.6231303 
Evaluando con validación cruzada: laplace = 0  usekernel = FALSE  adjust = 0.5 
RMSE con validación cruzada: 0.7472468 
Evaluando con validación cruzada: laplace = 1  usekernel = FALSE  adjust = 0.5 
RMSE con validación cruzada: 0.7472468 
Evaluando con validación cruzada: laplace = 2  usekernel = FALSE  adjust = 0.5 
RMSE con validación cruzada: 0.7472468 
Evaluando con validación cruzada: laplace = 0  usekernel = TRUE  adjust = 1 
RMSE con validación cruzada: 0.6305801 
Evaluando con validación cruzada: laplace = 1  usekernel = TRUE  adjust = 1 
RMSE con validación cruzada: 0.6305801 
Evaluando

Al utilizar validación cruzada para entrenar el modelo, el RMSE con validación cruzada fue 0.6231303, mientras que el RMSE sin validación cruzada fue 0.6245234. Aunque la diferencia en el RMSE es pequeña, el modelo con validación cruzada funcionó mejor, ya que obtuvo un RMSE más bajo. Además, la validación cruzada proporciona una evaluación más robusta y generalizable del modelo, ayudando a evitar el sobreajuste al evaluar el modelo en múltiples particiones del conjunto de datos.

In [22]:
# ====================================================
# Optimización de Modelos de Regresión y Clasificación
# ====================================================
library(e1071)
library(dplyr)
library(caret)
library(ggplot2)
library(rpart)
library(rpart.plot)
library(randomForest)

# Cargar datos preprocesados
train_data <- read.csv("train_preprocessed.csv", stringsAsFactors = TRUE)
test_data  <- read.csv("test_preprocessed.csv", stringsAsFactors = TRUE)

train_data$SalePrice <- as.numeric(as.character(train_data$SalePrice))
test_data$SalePrice  <- as.numeric(as.character(test_data$SalePrice))

# Revisar valores faltantes
if (sum(is.na(train_data)) > 0) {
  train_data <- na.omit(train_data)  # Eliminar filas con NA
}

if (sum(is.na(test_data)) > 0) {
  test_data <- na.omit(test_data)  # Eliminar filas con NA
}

# Asegurar consistencia en los factores
test_data <- test_data %>% mutate(across(where(is.factor), ~ factor(., levels = levels(train_data[[cur_column()]]))))

# Definir los predictores y la variable de salida
predictors <- setdiff(names(train_data), "SalePrice")

# ====================================================
# Creación de la Variable de Clasificación PriceCat
# ====================================================
# Usamos los cuartiles para definir los cortes:
#   - "Economicas": SalePrice < primer cuartil
#   - "Intermedias": primer cuartil <= SalePrice < tercer cuartil
#   - "Caras": SalePrice >= tercer cuartil

cat("\nResumen de SalePrice en train:\n")
print(summary(train_data$SalePrice))

# Calcular umbrales basados en cuartiles
cuartiles <- quantile(train_data$SalePrice, probs = c(0.25, 0.75), na.rm = TRUE)
lower_threshold <- cuartiles[1]
upper_threshold <- cuartiles[2]

cat("\nUmbrales elegidos para clasificar:\n")
cat("Economicas: < ", lower_threshold, "\n")
cat("Intermedias: [", lower_threshold, ", ", upper_threshold, ")\n")
cat("Caras: >= ", upper_threshold, "\n\n")

# Crear la variable categórica PriceCat en el conjunto de entrenamiento
train_data$PriceCat <- dplyr::case_when(
  train_data$SalePrice < lower_threshold ~ "Economicas",
  train_data$SalePrice < upper_threshold ~ "Intermedias",
  TRUE ~ "Caras"
)

# Asegurar que PriceCat esté presente también en el conjunto de test
test_data$PriceCat <- dplyr::case_when(
  test_data$SalePrice < lower_threshold ~ "Economicas",
  test_data$SalePrice < upper_threshold ~ "Intermedias",
  TRUE ~ "Caras"
)

# ====================================================
# Definir los bins
# ====================================================

# Definir número de bins
n_bins <- 50
unique_vals <- length(unique(train_data$SalePrice))
n_bins <- min(n_bins, unique_vals - 1)

# Crear los cortes usando cuantiles en train_data
bins <- quantile(train_data$SalePrice, probs = seq(0, 1, length.out = n_bins + 1), na.rm = TRUE)
bins <- unique(bins)  # Evitar cortes repetidos
train_data$SalePrice_bin <- cut(train_data$SalePrice, breaks = bins, include.lowest = TRUE, dig.lab = 10)

# Aplicar los mismos cortes a test_data
test_data$SalePrice_bin <- cut(test_data$SalePrice, breaks = bins, include.lowest = TRUE, dig.lab = 10)

# Calcular los centros de los bins
bin_centers <- (head(bins, -1) + tail(bins, -1)) / 2



# ====================================================
# 1. Ajuste de hiperparámetros en Naive Bayes (Regresión)
# ====================================================

grid_nb <- expand.grid(usekernel = c(TRUE, FALSE), fL = c(0, 1), adjust = c(0.5, 1))
results_nb <- data.frame()

for (i in 1:nrow(grid_nb)) {
  params <- grid_nb[i, ]

  # Entrenar el modelo con la variable categórica
  nb_model <- naiveBayes(SalePrice_bin ~ ., data = train_data,
                         usekernel = params$usekernel, fL = params$fL, adjust = params$adjust)

  # Obtener probabilidades para cada bin en el conjunto de prueba
  pred_probs <- predict(nb_model, newdata = test_data, type = "raw")

  # Calcular la predicción final como el valor esperado
  pred <- apply(pred_probs, 1, function(prob_vec) sum(prob_vec * bin_centers))

  # Calcular RMSE
  rmse <- RMSE(pred, test_data$SalePrice)
  results_nb <- rbind(results_nb, cbind(params, RMSE = rmse))
}

best_nb <- results_nb[which.min(results_nb$RMSE), ]

# ====================================================
# 2. Ajuste de hiperparámetros en Árbol de Regresión
# ====================================================

grid_tree <- expand.grid(maxdepth = c(3, 5), minsplit = c(5, 15))
results_tree <- data.frame()

for (i in 1:nrow(grid_tree)) {
  params <- grid_tree[i, ]
  tree_model <- rpart(SalePrice ~ ., data = train_data, method = "anova", control = rpart.control(maxdepth = params$maxdepth, minsplit = params$minsplit))
  pred <- predict(tree_model, newdata = test_data)
  pred <- as.numeric(pred)
  rmse <- RMSE(pred, test_data$SalePrice)
  results_tree <- rbind(results_tree, cbind(params, RMSE = rmse))
}

best_tree <- results_tree[which.min(results_tree$RMSE), ]

# ====================================================
# 3. Ajuste de hiperparámetros en Random Forest con Depuración
# ====================================================
set.seed(123)  # Para reproducibilidad

grid_rf <- expand.grid(mtry = c(2, 4, sqrt(length(predictors))), ntree = c(100, 300))
results_rf <- data.frame(mtry = numeric(), ntree = numeric(), RMSE = numeric())

for (i in 1:nrow(grid_rf)) {
  params <- grid_rf[i, ]

  # Entrenar modelo
  rf_model <- tryCatch({
    randomForest(SalePrice ~ ., data = train_data,
                 mtry = params$mtry, ntree = params$ntree, importance = TRUE)
  }, error = function(e) {
    return(NULL)
  })

  # Verificar si el modelo se entrenó correctamente
  if (is.null(rf_model)) next

  # Predicción
  pred <- tryCatch({
      pred <- predict(rf_model, newdata = test_data)

      # Eliminar valores NA en la predicción
      valid_idx <- !is.na(pred) & !is.na(test_data$SalePrice)
      pred <- pred[valid_idx]
      actual <- test_data$SalePrice[valid_idx]

      # Verificar que haya datos válidos antes de calcular RMSE
      if (length(pred) > 0) {
        rmse <- RMSE(pred, actual)
        results_rf <- rbind(results_rf, data.frame(mtry = params$mtry, ntree = params$ntree, RMSE = rmse))
      }
  }, error = function(e) {
    return(NULL)
  })

  # Verificar si la predicción es válida
  if (is.null(pred) || length(pred) == 0) next
}

best_rf <- results_rf[which.min(results_rf$RMSE), ]

# ====================================================
# 4. Ajuste de hiperparámetros en Naive Bayes (Clasificación)
# ====================================================

grid_nb_class <- expand.grid(usekernel = c(TRUE, FALSE), fL = c(0, 1), adjust = c(0.5, 1.5))
results_nb_class <- data.frame()

for (i in 1:nrow(grid_nb_class)) {
  params <- grid_nb_class[i, ]
  nb_model_class <- naiveBayes(PriceCat ~ ., data = train_data, usekernel = params$usekernel, fL = params$fL, adjust = params$adjust)
  pred_class <- predict(nb_model_class, newdata = test_data)
  acc <- mean(pred_class == test_data$PriceCat)
  results_nb_class <- rbind(results_nb_class, cbind(params, Accuracy = acc))
}

best_nb_class <- results_nb_class[which.max(results_nb_class$Accuracy), ]

# ====================================================
# Comparación de Resultados
# ====================================================

cat("Mejor modelo Naive Bayes (Regresión):\n")
if (nrow(best_nb) > 0) {
  print(best_nb)
} else {
  cat("No se encontraron resultados válidos para Naive Bayes (Regresión).\n")
}

cat("\nMejor modelo Árbol de Regresión:\n")
if (nrow(best_tree) > 0) {
  print(best_tree)
} else {
  cat("No se encontraron resultados válidos para Árbol de Regresión.\n")
}

cat("\nMejor modelo Random Forest:\n")
if (nrow(results_rf) > 0) {
  best_rf <- results_rf[which.min(results_rf$RMSE), ]
  print(best_rf)
} else {
  cat("\nError: No se generaron resultados válidos para Random Forest.\n")
}

cat("\nMejor modelo Naive Bayes (Clasificación):\n")
if (nrow(best_nb_class) > 0) {
  print(best_nb_class)
} else {
  cat("No se encontraron resultados válidos para Naive Bayes (Clasificación).\n")
}


Resumen de SalePrice en train:
    Min.  1st Qu.   Median     Mean  3rd Qu.     Max. 
-3.94029 -0.53110  0.03993  0.10218  0.71612  2.48721 

Umbrales elegidos para clasificar:
Economicas: <  -0.5310966 
Intermedias: [ -0.5310966 ,  0.7161206 )
Caras: >=  0.7161206 

Mejor modelo Naive Bayes (Regresión):
  usekernel fL adjust      RMSE
1      TRUE  0    0.5 0.6533122

Mejor modelo Árbol de Regresión:
  maxdepth minsplit     RMSE
1        3        5 0.169351

Mejor modelo Random Forest:
     mtry ntree      RMSE
3 10.0995   100 0.2191537

Mejor modelo Naive Bayes (Clasificación):
  usekernel fL adjust  Accuracy
1      TRUE  0    0.5 0.5539033


Los modelos mejoraron después de ajustar los hiperparámetros. El Árbol de Regresión mostró una mejora significativa en su rendimiento, con un RMSE reducido de 0.5227 a 0.1694. El Random Forest también mejoró, pasando de un RMSE de 0.3672 a 0.2192. Sin embargo, el Naive Bayes (Regresión) no experimentó una mejora considerable, con un RMSE ajustado de 0.6533, permaneciendo inferior en comparación con los otros modelos y el modelo de Clasificación Obtuvo un Accuracy de 0.5539. En general, el ajuste de hiperparámetros mejoró el rendimiento de los modelos no lineales, especialmente los árboles de decisión y los bosques aleatorios.